In [ ]:
import pandas as pd
import re
from jiwer import wer
import ast
import Levenshtein
import numpy as np
import ast

# normalization library
import unicodedata
import contractions
from num2words import num2words

# google cloud library
from googleapiclient import discovery
from google.auth import default

In [16]:
# For Excel files, use read_excel instead
all_datasets = pd.read_excel("results/all_result_processed_normalized.xlsx", index_col=False)

In [24]:
def analyze_medical_text(project_id, location, text_content):
    """
    Call Google Healthcare API to analyze medical entities in text.
    Returns full API response payload for downstream mention reconstruction.
    """
    try:
        credentials, _ = default()
        service = discovery.build("healthcare", "v1", credentials=credentials)

        nlp_service_name = f"projects/{project_id}/locations/{location}/services/nlp"
        body = {"documentContent": text_content}

        response = service.projects().locations().services().nlp().analyzeEntities(
            nlpService=nlp_service_name,
            body=body,
        ).execute()

        return response
    except Exception as e:
        print(f"Error calling healthcare API: {e}")
        return {}


def _collect_mentions_from_response(response):
    """
    Extract mention spans in a normalized shape:
    {begin, end, replacement}
    """
    if not response:
        return []

    entities = response.get("entities", []) or []
    entity_mentions = response.get("entityMentions", []) or []

    # Map entity IDs to preferred terms from the entity catalog.
    id_to_term = {}
    for entity in entities:
        entity_id = entity.get("entityId")
        preferred_term = entity.get("preferredTerm")
        if entity_id and preferred_term:
            id_to_term[entity_id] = preferred_term

    mentions_to_replace = []

    # Preferred path: top-level entityMentions usually contains reliable offsets.
    if entity_mentions:
        for mention in entity_mentions:
            text_obj = mention.get("text", {}) or {}
            surface_text = text_obj.get("content", "")
            begin_offset = text_obj.get("beginOffset")

            if begin_offset is None or not surface_text:
                continue

            linked_entities = mention.get("linkedEntities", []) or []
            linked_id = None
            if linked_entities:
                linked_id = linked_entities[0].get("entityId")

            # Fall back to mention-level entityId if linkedEntities is missing.
            entity_id = linked_id or mention.get("entityId")
            if not entity_id:
                continue

            preferred_term = id_to_term.get(entity_id, surface_text)
            end_offset = begin_offset + len(surface_text)
            replacement_tag = f"[{entity_id}: {preferred_term}]"

            mentions_to_replace.append(
                {
                    "begin": begin_offset,
                    "end": end_offset,
                    "replacement": replacement_tag,
                }
            )

        return mentions_to_replace

    # Fallback path: some responses only include entity->mentions.
    for entity in entities:
        entity_id = entity.get("entityId", "")
        preferred_term = entity.get("preferredTerm", "")
        if not entity_id or not preferred_term:
            continue

        mentions = entity.get("mentions", []) or []
        for mention in mentions:
            text_obj = mention.get("text", {}) or {}
            surface_text = text_obj.get("content", "")
            begin_offset = text_obj.get("beginOffset")

            if begin_offset is None or not surface_text:
                continue

            end_offset = begin_offset + len(surface_text)
            replacement_tag = f"[{entity_id}: {preferred_term}]"
            mentions_to_replace.append(
                {
                    "begin": begin_offset,
                    "end": end_offset,
                    "replacement": replacement_tag,
                }
            )

    return mentions_to_replace


def reconstruct_text_with_ner_tags(text_content, response):
    """
    Reconstruct the original text with inline NER tags.
    Replaces mention spans with format: [UMLS/C0000726: Abdomen]
    """
    if pd.isna(text_content) or text_content is None:
        return text_content

    text_content = str(text_content)
    mentions_to_replace = _collect_mentions_from_response(response)
    if not mentions_to_replace:
        return text_content

    # Sort by reverse offset to avoid shifting indices during replacement.
    mentions_to_replace.sort(key=lambda x: x["begin"], reverse=True)

    # Skip overlapping spans to prevent malformed replacements.
    reconstructed_text = text_content
    last_begin = len(text_content) + 1
    for mention in mentions_to_replace:
        begin = mention["begin"]
        end = mention["end"]
        replacement = mention["replacement"]

        if begin < 0 or end > len(reconstructed_text) or begin >= end:
            continue

        if end > last_begin:
            continue

        reconstructed_text = (
            reconstructed_text[:begin] + replacement + reconstructed_text[end:]
        )
        last_begin = begin

    return reconstructed_text


def apply_ner_to_row(text_content, project_id, location):
    """
    Apply NER analysis to one transcript row and return the tagged transcript.
    """
    if pd.isna(text_content) or text_content is None:
        return text_content

    text_content = str(text_content)
    if not text_content.strip():
        return text_content

    response = analyze_medical_text(project_id, location, text_content)
    return reconstruct_text_with_ner_tags(text_content, response)

In [25]:
# Apply NER processing to create norm_human_transcript_ner column
PROJECT_ID = "bio-ramp-ner"
LOCATION = "us-central1"

print(f"Processing {len(all_datasets)} rows for NER tagging...")
print("This may take a while depending on the number of rows and text length.\n")

# Use apply with lambda to process each row
all_datasets['norm_human_transcript_ner'] = all_datasets['norm_human_transcript'].apply(
    lambda text: apply_ner_to_row(text, PROJECT_ID, LOCATION)
)

print("\n✓ NER processing complete!")
print(f"Created column 'norm_human_transcript_ner' with {len(all_datasets)} rows")
print("\n" + "=" * 70)
print("Sample of first 3 rows:")
print("=" * 70)

# Display sample output
for idx in range(min(3, len(all_datasets))):
    print(f"\nRow {idx}:")
    print(f"Original:  {all_datasets['norm_human_transcript'].iloc[idx][:100]}...")
    print(f"NER Tagged: {all_datasets['norm_human_transcript_ner'].iloc[idx][:100]}...")

Processing 120 rows for NER tagging...
This may take a while depending on the number of rows and text length.


✓ NER processing complete!
Created column 'norm_human_transcript_ner' with 120 rows

Sample of first 3 rows:

Row 0:
Original:  good morning i am doctor smith from babylon can you just confirm your name date of birth and the fir...
NER Tagged: good morning i am doctor smith from [UMLS/C0039705: Tetrodotoxin] can you just confirm your name dat...

Row 1:
Original:   hello hi i am doctor jacob and welcome to babylon hi  hi so just before we start is it alright if y...
NER Tagged:  hello hi i am doctor jacob and welcome to babylon hi  hi so just before we start is it alright if y...

Row 2:
Original:  hi there good morning hello good morning  i am doctor deen mirza from gp at hand nice to see you nic...
NER Tagged: hi there good morning hello good morning  i am doctor deen mirza from gp at hand nice to see you nic...


In [23]:
# Quick validation on first 2 rows before full batch run
sample_df = all_datasets.head(2).copy()
sample_df["norm_human_transcript_ner"] = sample_df["norm_human_transcript"].apply(
    lambda text: apply_ner_to_row(text, PROJECT_ID, LOCATION)
)

for i, row in sample_df.iterrows():
    tagged_text = row["norm_human_transcript_ner"]
    has_tag = "[UMLS/" in str(tagged_text)
    print(f"Row {i} has inline UMLS tags: {has_tag}")
    print(str(tagged_text)[:220])
    print("-" * 80)

Row 0 has inline UMLS tags: True
good morning i am doctor smith from [UMLS/C0039705: Tetrodotoxin] can you just confirm your name date of birth and the first line of your address please hi my name is [UMLS/C1256210: Rudbeckia hirta, black eyed Susan, fl
--------------------------------------------------------------------------------
Row 1 has inline UMLS tags: True
 hello hi i am doctor jacob and welcome to babylon hi  hi so just before we start is it alright if you could confirm your name for me please yep  john doe okay and your date of birth  uhh  twentyone twelve and nineteen  
--------------------------------------------------------------------------------


In [ ]:
 hello hi i am doctor jacob and welcome to babylon hi  hi so just before we start is it alright if you could confirm your name for me please yep  john doe okay and your date of birth  uhh  twentyone twelve and nineteen  eightysix and your address for me please is  number one london street  nw three six pq  that is correct and just to confirm that you are in a secure location and we can have a confidential conversation we are yes okay so i understand you are having a  a [UMLS/C0018681: Headache] is that correct  yeah iive had a  a sort of a a a very sort of bad [UMLS/C0205082: Severe (severity modifier)] [UMLS/C0018681: Headache] the past four days  and  i have been sort of feeling a bit hot as well  i have got got a bit of a [UMLS/C0039810: thermography] i think  i have not measured it but  think i have yeah i think i have got a [UMLS/C0015967: Fever] okay right and  so what started first was it the [UMLS/C0018681: Headache] or the [UMLS/C0015967: Fever]  so  [UMLS/C0018681: Headache] started  first  and then sort of yeah [UMLS/C0015967: Fever] sort of came along a bit afterwards okay afterwards right and what have you taken so far for your [UMLS/C0018681: Headache]  so i have had some [UMLS/C0000970: acetaminophen] and some [UMLS/C0020740: ibuprofen] but it is not really helping at the moment  okay so just a few more questions with regards to your [UMLS/C0018681: Headache] so do you find it difficult to look at the light yep  no  like  it it is fine light does not seem to bother me too much okay  and is it any [UMLS/C0030193: Pain] when you try to move your [UMLS/C0027530: Neck] from side to side look up and down is there any [UMLS/C0030193: Pain] and i have got some general [UMLS/C0030193: Pain] [UMLS/C0026845: Muscle Tissue]  not necessarily sort of in my [UMLS/C0027530: Neck] it is kind of a bit all over really okay all over right and have you been [UMLS/C0221423: Illness (finding)] with this  [UMLS/C0018681: Headache] no no i have not i have not been [UMLS/C0221423: Illness (finding)] okay and do you have a [UMLS/C0010200: Coughing]  no no [UMLS/C0010200: Coughing] okay so there is no [UMLS/C0010200: Coughing] or [UMLS/C0024117: Chronic Obstructive Airway Disease] yeah  no no okay and you your [UMLS/C0013443: Ear structure] are okay there is no [UMLS/C0013456: Earache]  no it is it seems to be okay it it is fine okay all right and with this [UMLS/C0018681: Headache] have you experienced any [UMLS/C1168556: Pins - Internal fixators] and [UMLS/C0027551: Needle device] in your [UMLS/C0015450: Face] [UMLS/C0446516: Upper arm] and [UMLS/C1140621: Leg]  no no [UMLS/C1168556: Pins - Internal fixators] and [UMLS/C0085178: Needlestick Injuries] any [UMLS/C0026845: Muscle Tissue] [UMLS/C0151786: Muscle Weakness] at all  just i i suppose general [UMLS/C0030193: Pain] [UMLS/C0444584: Whole body] [UMLS/C0030193: Pain] not not any [UMLS/C0004093: Asthenia] but where you could not you know hold a cup or anything like that that is what i mean  no i am i am sort of yeah  seem to be sort of fine with that sort of stuff but yeah oh  okay and  just a few more questions to you because i do not have any of your background history  do you suffer from any [UMLS/C4745084: Medical Condition]  no  general medical history is fine fine and no surgical history  then  and no [UMLS/C5441521: Complaint (finding)] yeah  no  i i suppose  i have been away to   thailand    week or so ago   had some like  usual [UMLS/C4520766: Bite of mosquito] from sort of traveling around and  a friend of mine has also been a bit unwell but   yeah that s the only sort of like history per se right and  do you have any kind of [UMLS/C0036973: Shivering] at all do you [UMLS/C0036973: Shivering] with the [UMLS/C0015967: Fever]  i   i i do feel [UMLS/C0024117: Chronic Obstructive Airway Disease] quite a lot  you know it is  i no sort of [UMLS/C0036973: Shivering] but definitely like a bit [UMLS/C0024117: Chronic Obstructive Airway Disease] yeah okay right and are you using  you mentioned about the [UMLS/C0000970: acetaminophen] and  now but are you on any  regular any [UMLS/C0013227: Pharmaceutical Preparations] anything that you are using at all yep  no okay  any [UMLS/C0020517: Hypersensitivity] at all  no  alright okay and okay so  i i believe that it is quite important for us to actually examine you yeah  it it is not  possible for us to do that online i think you would need to come into one of our hubs to have a proper examination and i i think you would need one quite urgently okay so the thing that is  running in the back of my mind are  kind of a [UMLS/C5826884: History of recent viral illness] that can one can get with  sort of [UMLS/C4520766: Bite of mosquito] or foreign travel okay like    the other common thing around thailand it will be like some dengue but that would have like you would have some  [UMLS/C4732730: Blood spots] or [UMLS/C0019080: Hemorrhage] or something which you do not have at the moment do you okay   no i mean i have got a bit of a [UMLS/C0015230: Exanthema] but nothing like  no no no no [UMLS/C0019080: Hemorrhage] okay so where is the [UMLS/C0015230: Exanthema]  it is  the [UMLS/C0015230: Exanthema] is kind of  all over my [UMLS/C0444584: Whole body] like it is not it is not it is not really localised how would you describe how would you describe the [UMLS/C0015230: Exanthema]   sort of [UMLS/C0235267: Redness of eye] and [UMLS/C0033774: Pruritus] and [UMLS/C0033774: Pruritus] okay so if you were to put a glass on top of the [UMLS/C0015230: Exanthema] does that [UMLS/C0015230: Exanthema] go away yeah  yes it goes away okay okay so  what we need to do  need to urgently examine you and that is sometime today okay and  after our [UMLS/C0582103: Medical Examination] there is a high possibility that sometimes we might need to send you to hospital but i am not sure as yet so i would like you to be seen in one of our hubs as soon as possible  okay okay so  i will  recommend that you would need to be urgently seen by the doctor preferably in the next few hours time okay and  in any case if that is not available i would like you to take yourself to  alright any questions at all okay okay  no sounds  serious  you know obviously when i have spoken to you you you did not tick the box for many theories but you are in the middle somewhere let us put it that way okay   you know things like we have to rule out is [UMLS/C0025289: Meningitis] and all but ben you just have a general [UMLS/C0026845: Muscle Tissue] [UMLS/C0030193: Pain] you did not say that you had things like you had [UMLS/C0033213: Problem] looking at the light or something  those are things when you think of [UMLS/C0025289: Meningitis] but and then it is the  in [UMLS/C0025289: Meningitis] you get a [UMLS/C0015230: Exanthema] while in any kind of [UMLS/C0042776: Virus]  the [UMLS/C0015230: Exanthema] usually goes away saying that we do not know whether these are the beginning stage so that is why i would rather but i would rather you see a  gp as soon as possible if  we have got a slot available but if not i think you need to be seen and maybe have some [UMLS/C0018941: Hematologic Tests] and [UMLS/C0582103: Medical Examination] okay okay cool thank you very much all right you take care ben bye now cheers bye  bye

In [ ]:
 hello hi i am doctor jacob and welcome to babylon hi  hi so just before we start is it alright if you could confirm your name for me please yep  john doe okay and your date of birth  uhh  twentyone twelve and nineteen  eightysix and your address for me please is  number one london street  nw three six pq  that is correct and just to confirm that you are in a secure location and we can have a confidential conversation we are yes okay so i understand you are having a  a headache is that correct  yeah iive had a  a sort of a a a very sort of bad severe headache the past four days  and  i have been sort of feeling a bit hot as well  i have got got a bit of a temperature i think  i have not measured it but  think i have yeah i think i have got a fever okay right and  so what started first was it the headache or the fever  so  headache started  first  and then sort of yeah fever sort of came along a bit afterwards okay afterwards right and what have you taken so far for your headache  so i have had some paracetamol and some ibuprofen but it is not really helping at the moment  okay so just a few more questions with regards to your headache so do you find it difficult to look at the light yep  no  like  it it is fine light does not seem to bother me too much okay  and is it any pain when you try to move your neck from side to side look up and down is there any pain and i have got some general aching muscles  not necessarily sort of in my neck it is kind of a bit all over really okay all over right and have you been sick with this  headache no no i have not i have not been sick okay and do you have a cough  no no cough okay so there is no cough or cold yeah  no no okay and you your ears are okay there is no earaches  no it is it seems to be okay it it is fine okay all right and with this headache have you experienced any pins and needles in your face arms and legs  no no pins and needles any muscle weaknesses at all  just i i suppose general aching body aching not not any weakness but where you could not you know hold a cup or anything like that that is what i mean  no i am i am sort of yeah  seem to be sort of fine with that sort of stuff but yeah oh  okay and  just a few more questions to you because i do not have any of your background history  do you suffer from any medical conditions  no  general medical history is fine fine and no surgical history  then  and no medical surgical complaints yeah  no  i i suppose  i have been away to   thailand    week or so ago   had some like  usual mosquito bites from sort of traveling around and  a friend of mine has also been a bit unwell but   yeah that s the only sort of like history per se right and  do you have any kind of shivering at all do you shivering with the fever  i   i i do feel cold quite a lot  you know it is  i no sort of shivering but definitely like a bit cold  cold yeah okay right and are you using  you mentioned about the paracetamol and  now but are you on any  regular any medication anything that you are using at all yep  no okay  any allergies at all  no  alright okay and okay so  i i believe that it is quite important for us to actually examine you yeah  it it is not  possible for us to do that online i think you would need to come into one of our hubs to have a proper examination and i i think you would need one quite urgently okay so the thing that is  running in the back of my mind are  kind of a viral illnesses that can one can get with  sort of mosquito bites or foreign travel okay like    the other common thing around thailand it will be like some dengue but that would have like you would have some  blood spots or bleeding or something which you do not have at the moment do you okay   no i mean i have got a bit of a rash but nothing like  no no no no bleeding okay so where is the rash  it is  the rash is kind of  all over my body like it is not it is not it is not really localised how would you describe how would you describe the rash   sort of red and itchy red and itchy okay so if you were to put a glass on top of the rash does that rash go away yeah  yes it goes away okay okay so  what we need to do  need to urgently examine you and that is sometime today okay and  after our examination there is a high possibility that sometimes we might need to send you to hospital but i am not sure as yet so i would like you to be seen in one of our hubs as soon as possible  okay okay so  i will  recommend that you would need to be urgently seen by the doctor preferably in the next few hours time okay and  in any case if that is not available i would like you to take yourself to  alright any questions at all okay okay  no sounds  serious  you know obviously when i have spoken to you you you did not tick the box for many theories but you are in the middle somewhere let us put it that way okay   you know things like we have to rule out is meningitis and all but ben you just have a general muscle ache you did not say that you had things like you had problems looking at the light or something  those are things when you think of meningitis but and then it is the  in meningitis you get a nonblanching  rash while in any kind of virus  the rash usually goes away saying that we do not know whether these are the beginning stage so that is why i would rather but i would rather you see a  gp as soon as possible if  we have got a slot available but if not i think you need to be seen and maybe have some blood tests and examinations okay okay cool thank you very much all right you take care ben bye now cheers bye  bye